# structured-v15 dev-13

Launch this on Kaggle. Do not submit it to the leaderboard.

It runs `agents/structured-v15` on the 13-task dev cohort. Three of those tasks
are environment-unreachable (`fastapi_14246`, `fastapi_14262`, `fastapi_14356`),
so the reachable ceiling is 10. The notebook keeps traces and prints resolved
count, per-task wall time, no-edit runs, and calls to the first edit.

The official compiler attaches a callback only when the host registry has that
name. The next code cell registers `callbacks.py` before `compile_submission`.
Run that cell before the evaluation loop imports the harness runner.


In [ ]:
import os
import sys
from pathlib import Path

TASK_IDS = [
    "fastapi_11194",
    "fastapi_14246",
    "fastapi_14262",
    "fastapi_14356",
    "fastapi_14786",
    "fastapi_14794",
    "rich_3105",
    "rich_3454",
    "rich_3472",
    "rich_3934",
    "rich_3938",
    "rich_3944",
    "rich_4070",
]
REACHABLE_CEILING = 10
AGENT_DIR = Path(os.environ.get("AGENT_DIR", "/kaggle/working/agent"))
if not (AGENT_DIR / "callbacks.py").is_file():
    AGENT_DIR = Path("agents/structured-v15").resolve()
sys.path.insert(0, str(AGENT_DIR))
os.environ.setdefault("GEMMA_CHECK_ROOT", "/workspace")
print("structured-v15", AGENT_DIR)
print("tasks", len(TASK_IDS), "reachable ceiling", REACHABLE_CEILING)

In [ ]:
from callbacks import register_callbacks


def install_v15_callbacks():
    """Register the candidate callbacks on the compiler the runner calls."""
    import adk_submission
    from adk_submission import CallbackRegistry

    original = adk_submission.compile_submission

    def wrapped(*args, **kwargs):
        registry = kwargs.get("callback_registry")
        if registry is None:
            registry = CallbackRegistry()
            kwargs["callback_registry"] = registry
        register_callbacks(registry)
        return original(*args, **kwargs)

    adk_submission.compile_submission = wrapped
    try:
        import swegemma.harness.agent_runner as runner

        runner.compile_submission = wrapped
    except Exception:
        pass
    return wrapped


install_v15_callbacks()
print("structured-v15 callbacks registered")

## After the official loop

Point `RESULTS` at the run directory that holds `task_results.jsonl` and
`results/traces/`. This cell does not start the model. Traces stay on disk.


In [ ]:
import json
import os
from pathlib import Path

from callbacks import dev13_report

RESULTS = Path(os.environ.get("V15_RESULTS", "/kaggle/working"))
rows = []
results_path = RESULTS / "task_results.jsonl"
if results_path.is_file():
    for line in results_path.read_text().splitlines():
        if line.strip():
            rows.append(json.loads(line))
traces = {}
trace_dir = RESULTS / "results" / "traces"
if not trace_dir.is_dir():
    trace_dir = RESULTS / "traces"
if trace_dir.is_dir():
    for path in sorted(trace_dir.glob("*.json")):
        payload = json.loads(path.read_text())
        task_id = payload.get("instance_id") or path.stem.removeprefix("trace_")
        traces[task_id] = payload
report = dev13_report(rows, traces)
print(
    "resolved",
    report["resolved"],
    "of",
    report["n"],
    "; reachable",
    report["reachable_resolved"],
    "of",
    report["reachable_ceiling"],
)
print("no_edit_runs", report["no_edit_runs"])
for item in report["tasks"]:
    print(
        item["instance_id"],
        "resolved" if item["resolved"] else "unresolved",
        "wall",
        item["wall_seconds"],
        "no_edit" if item["no_edit"] else "edit",
        "calls_to_first_edit",
        item["calls_to_first_edit"],
    )
out = RESULTS / "structured-v15-dev13-report.json"
out.write_text(json.dumps(report, indent=2) + "\n")
print("wrote", out)